# Pendulo simples (didatico)

Fluxo:
1. Resolver com Euler
2. Mostrar saida numerica
3. Plotar graficos
4. Repetir com RK4


EDO: `theta'' + (g/L)*sin(theta) = 0`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# BLOCO 1: PARAMETROS
g = 9.81
L = 1.0
theta0 = np.deg2rad(20.0)
omega0 = 0.0
dt = 0.01
t_final = 20.0
t = np.arange(0.0, t_final + dt, dt)
N = len(t)

# BLOCO 2: EULER
def simular_euler():
    theta = np.zeros(N)
    omega = np.zeros(N)
    theta[0] = theta0
    omega[0] = omega0
    for i in range(N - 1):
        alpha = -(g / L) * np.sin(theta[i])
        omega[i + 1] = omega[i] + alpha * dt
        theta[i + 1] = theta[i] + omega[i] * dt
    return theta, omega

# BLOCO 3: RK4
def derivadas(theta, omega):
    return omega, -(g / L) * np.sin(theta)

def simular_rk4():
    theta = np.zeros(N)
    omega = np.zeros(N)
    theta[0] = theta0
    omega[0] = omega0
    for i in range(N - 1):
        th, om = theta[i], omega[i]
        k1_th, k1_om = derivadas(th, om)
        k2_th, k2_om = derivadas(th + 0.5 * dt * k1_th, om + 0.5 * dt * k1_om)
        k3_th, k3_om = derivadas(th + 0.5 * dt * k2_th, om + 0.5 * dt * k2_om)
        k4_th, k4_om = derivadas(th + dt * k3_th, om + dt * k3_om)
        theta[i + 1] = th + (dt / 6.0) * (k1_th + 2 * k2_th + 2 * k3_th + k4_th)
        omega[i + 1] = om + (dt / 6.0) * (k1_om + 2 * k2_om + 2 * k3_om + k4_om)
    return theta, omega

# BLOCO 4: FUNCOES DE APOIO
def mostrar_saida(nome, theta, omega):
    x = L * np.sin(theta)
    print(f"\nSaida numerica - {nome}")
    print(" i   t(s)    theta(rad)    omega(rad/s)    x(m)")
    for i in range(10):
        print(f"{i:2d}  {t[i]:6.3f}   {theta[i]:10.6f}    {omega[i]:11.6f}   {x[i]:8.5f}")

def plotar(nome, theta):
    x = L * np.sin(theta)
    y = -L * np.cos(theta)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    axs[0].plot(t, theta, color='tab:blue')
    axs[0].set_title(f'{nome} - theta(t)')
    axs[0].set_xlabel('tempo (s)')
    axs[0].set_ylabel('theta (rad)')
    axs[0].grid(alpha=0.3)

    axs[1].plot(x, y, color='tab:orange')
    axs[1].scatter([0.0], [0.0], color='black', s=30, label='pivo')
    axs[1].set_title(f'{nome} - trajetoria x-y')
    axs[1].set_xlabel('x (m)')
    axs[1].set_ylabel('y (m)')
    axs[1].axis('equal')
    axs[1].grid(alpha=0.3)
    axs[1].legend()

    plt.tight_layout()
    plt.show()

def animar_pendulo(nome, theta, passo=2):
    t_anim = t[::passo]
    theta_anim = theta[::passo]

    fig, (ax_p, ax_g) = plt.subplots(1, 2, figsize=(11, 4.5))
    fig.suptitle(f'{nome} - Animacao do pendulo')

    ax_p.set_aspect('equal')
    ax_p.set_xlim(-1.2 * L, 1.2 * L)
    ax_p.set_ylim(-1.2 * L, 0.2 * L)
    ax_p.set_title('Movimento')
    ax_p.plot(0, 0, 'ko', ms=6)
    haste, = ax_p.plot([], [], lw=2, color='tab:blue')
    bob, = ax_p.plot([], [], 'o', ms=10, color='tab:red')

    amp = max(np.max(np.abs(theta_anim)) * 1.2, 0.2)
    ax_g.set_xlim(0, t_final)
    ax_g.set_ylim(-amp, amp)
    ax_g.set_title('Angulo ao longo do tempo')
    ax_g.set_xlabel('tempo (s)')
    ax_g.set_ylabel('theta (rad)')
    linha_theta, = ax_g.plot([], [], color='tab:orange')

    def init():
        haste.set_data([], [])
        bob.set_data([], [])
        linha_theta.set_data([], [])
        return haste, bob, linha_theta

    def update(i):
        th = theta_anim[i]
        x = L * np.sin(th)
        y = -L * np.cos(th)
        haste.set_data([0, x], [0, y])
        bob.set_data([x], [y])
        linha_theta.set_data(t_anim[: i + 1], theta_anim[: i + 1])
        return haste, bob, linha_theta

    ani = FuncAnimation(fig, update, frames=len(theta_anim), init_func=init, interval=20, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())


In [ ]:
# BLOCO 5: EXECUCAO - EULER
th_euler, om_euler = simular_euler()
mostrar_saida('Euler', th_euler, om_euler)
plotar('Euler', th_euler)
animar_pendulo('Euler', th_euler)


In [ ]:
# BLOCO 6: EXECUCAO - RK4
th_rk4, om_rk4 = simular_rk4()
mostrar_saida('Runge-Kutta 4', th_rk4, om_rk4)
plotar('Runge-Kutta 4', th_rk4)
animar_pendulo('Runge-Kutta 4', th_rk4)
